In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "nyc_taxi.bronze.raw_trips"
SILVER_TABLE = "nyc_taxi.silver.trips_clean"

In [0]:
df = spark.read.table(BRONZE_TABLE)
print(f"BRONZE ROWS: {df.count()}")

BRONZE ROWS: 12885358


In [0]:
df = df.filter(
    (F.col("tpep_pickup_datetime") >= "2025-04-01") &
    (F.col("tpep_pickup_datetime") < "2025-07-01")
)
print(f"After date filter: {df.count()}")

After date filter: 12885353


In [0]:
df = df.filter(
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")) &
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") > 0) &
    (F.col("total_amount") > 0)
)
print(f"After validity filter: {df.count()}")

After validity filter: 11634226


In [0]:
df = df.withColumn(
    "congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0))
).withColumn(
    "Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0))
).withColumn(
    "store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("N"))
)

In [0]:
df = df.withColumn(
    "trip_duration_minutes",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
).withColumn(
    "tip_percentage",
    F.when(F.col("fare_amount") > 0, (F.col("tip_amount") / F.col("fare_amount")) * 100)
    .otherwise(0.0)
).withColumn(
    "pickup_hour", F.hour("tpep_pickup_datetime")
).withColumn(
    "pickup_day_of_week", F.dayofweek("tpep_pickup_datetime")
).withColumn(
    "has_valid_passenger_count", F.col("passenger_count") > 0
)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_TABLE)

print(f"Silver Table Rows Count {df.count()}")
print(F"Silver Table Created: {SILVER_TABLE}")

Silver Table Rows Count 11634226
Silver Table Created: nyc_taxi.silver.trips_clean
